# 03 - Baseline Random Forest

En este notebook se entrena una primera iteracion del modelo usando **Random Forest**.

El objetivo no es cerrar el modelo final, sino tener una referencia inicial mas interesante que un modelo lineal para explicar la aproximacion al problema de Machine Learning.


## 0. Importacion de librerias y configuracion


In [ ]:
from pathlib import Path
import pickle

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET = "is_canceled"

TRAIN_PATH = Path("../data/train/train.csv")
TEST_PATH = Path("../data/test/test.csv")
MODELS_DIR = Path("../models")

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 100)


## 1. Carga de train y test

Cargo los archivos generados a partir del dataset limpio. El split ya esta hecho de forma estratificada, por lo que mantiene la proporcion de reservas canceladas y no canceladas en train y test.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train: {train_df.shape[0]:,} filas y {train_df.shape[1]} columnas".replace(",", "."))
print(f"Test: {test_df.shape[0]:,} filas y {test_df.shape[1]} columnas".replace(",", "."))

train_df.head()


In [ ]:
target_summary = pd.DataFrame({
    "train_%": (train_df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2),
    "test_%": (test_df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2),
})

target_summary.index = ["No cancelada", "Cancelada"]
target_summary


## 2. Separacion de variables predictoras y target

La variable objetivo es `is_canceled`. El resto de columnas se usan como variables predictoras.


In [ ]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print(f"Variables predictoras: {X_train.shape[1]}")


## 3. Preprocesado

Random Forest trabaja bien con variables numericas, pero las variables categoricas deben transformarse.

- Numericas: imputacion con mediana.
- Categoricas: imputacion con moda y One-Hot Encoding.

No aplico escalado porque Random Forest no lo necesita para funcionar correctamente.


In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print(f"Variables numericas: {len(numeric_features)}")
print(f"Variables categoricas: {len(categorical_features)}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])


## 4. Entrenamiento del modelo baseline

Uso Random Forest como primera iteracion porque es un modelo robusto para datos tabulares y puede capturar relaciones no lineales entre variables.

Los parametros son iniciales; no se realiza todavia hiperparametrizacion.


In [ ]:
rf_baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

rf_baseline.fit(X_train, y_train)


## 5. Evaluacion del modelo

Evaluo el modelo sobre el conjunto de test. Las metricas principales son `precision`, `recall`, `f1` y `roc_auc`, ya que el objetivo es detectar cancelaciones sin generar demasiadas falsas alarmas.


In [ ]:
y_pred = rf_baseline.predict(X_test)
y_proba = rf_baseline.predict_proba(X_test)[:, 1]

metrics_df = pd.DataFrame([
    {
        "modelo": "Random Forest baseline",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
])

metrics_df.round(4)


In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["No cancelada", "Cancelada"],
    zero_division=0,
))


### Matriz de confusion

La matriz de confusion permite ver cuantos casos se clasifican correctamente y cuantos errores comete el modelo en cada clase.


In [ ]:
confusion_matrix_rf = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
ax = sns.heatmap(
    confusion_matrix_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No cancelada", "Cancelada"],
    yticklabels=["No cancelada", "Cancelada"],
)
ax.set_title("Matriz de confusion - Random Forest baseline")
ax.set_xlabel("Prediccion")
ax.set_ylabel("Valor real")
plt.tight_layout()
plt.show()


## 6. Guardado del modelo baseline

Guardo el pipeline completo como `baseline_random_forest.pkl`, incluyendo preprocesado y modelo. Asi queda claramente separado del modelo final de XGBoost.


In [ ]:
MODELS_DIR.mkdir(exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_random_forest.pkl"

with open(baseline_model_path, "wb") as file:
    pickle.dump(rf_baseline, file)

print(f"Modelo baseline guardado en: {baseline_model_path}")


## 7. Conclusiones

- Se ha entrenado una primera iteracion con Random Forest.
- El modelo utiliza el dataset limpio y el split train/test generado previamente.
- El preprocesado se integra dentro de un `Pipeline`, evitando aplicar transformaciones manuales fuera del entrenamiento.
- Este modelo sirve como baseline inicial para comparar posteriormente con otros modelos e hiperparametrizacion.
